# Diffusion Policy Training on OpenCabinet (Colab)

**Requirements:** Colab Pro with A100 GPU recommended. Select GPU runtime before running.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

DRIVE_DIR = '/content/drive/MyDrive/cs188_diffusion_policy'
os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. Clone repo and run install.sh

In [ ]:
!apt-get update -qq && apt-get install -y -qq git cmake build-essential python3-dev libosmesa6-dev libglew-dev libgl1 linux-libc-dev python3-venv 2>&1 | tail -1

![ -d /content/cs188-cabinet-door-project ] || git clone https://github.com/VC-cyber/cs188-cabinet-door-project.git /content/cs188-cabinet-door-project

!cd /content/cs188-cabinet-door-project && yes y | bash ./install.sh

## 3. Install diffusion_policy + its dependencies

In [ ]:
%env PROJECT=/content/cs188-cabinet-door-project

!source $PROJECT/.venv/bin/activate && pip install -q -e $PROJECT/cabinet_door_project/diffusion_policy
!source $PROJECT/.venv/bin/activate && pip install -q --no-deps 'robomimic @ git+https://github.com/ARISE-Initiative/robomimic.git@master'
!source $PROJECT/.venv/bin/activate && pip install -q hydra-core zarr einops diffusers wandb tqdm accelerate scipy h5py numba imagecodecs numcodecs dill av pandas tensorboardX numpydantic pyarrow
!echo 'Done.'

## 4. Download OpenCabinet dataset

In [ ]:
!cd /content/cs188-cabinet-door-project && source .venv/bin/activate && cd cabinet_door_project && python 04_download_dataset.py

## 5. Patch robomimic + diffusion_policy

Upstream robomimic 0.5 is missing `LANG_EMB_KEY` and `LangEncoder` that the diffusion_policy repo expects. Also fixes a `diffusers` import compatibility issue.

In [ ]:
import os, subprocess

PROJECT = '/content/cs188-cabinet-door-project'
VENV_PY = f'{PROJECT}/.venv/bin/python'
VENV_SITE = subprocess.check_output(
    [VENV_PY, '-c', 'import site; print(site.getsitepackages()[0])']
).decode().strip()
DP = f'{PROJECT}/cabinet_door_project/diffusion_policy/diffusion_policy'

print(f'Site-packages: {VENV_SITE}')

# --- Patch 1: Add LANG_EMB_KEY to robomimic/macros.py ---
p = f'{VENV_SITE}/robomimic/macros.py'
with open(p) as f: src = f.read()
if 'LANG_EMB_KEY' not in src:
    with open(p, 'w') as f:
        f.write(src.replace('WANDB_API_KEY = None', 'WANDB_API_KEY = None\n\nLANG_EMB_KEY = "lang_emb"'))
    print('Patched macros.py')
else:
    print('macros.py: OK')

# --- Patch 2: Add LangEncoder to lang_utils.py ---
p = f'{VENV_SITE}/robomimic/utils/lang_utils.py'
with open(p) as f: src = f.read()
if 'class LangEncoder' not in src:
    src += '''

class LangEncoder:
    def __init__(self, device=None):
        self.device = device

    def get_lang_emb(self, langs):
        if isinstance(langs, str):
            langs = [langs]
        tokens = tz(
            text=langs, add_special_tokens=True, max_length=25,
            padding="max_length", return_attention_mask=True, return_tensors="pt",
        )
        return lang_emb_model(**tokens)["text_embeds"].detach()
'''
    with open(p, 'w') as f: f.write(src)
    print('Patched lang_utils.py')
else:
    print('lang_utils.py: OK')

# --- Patch 3: Fix lr_scheduler.py diffusers import ---
p = f'{DP}/model/common/lr_scheduler.py'
with open(p) as f: src = f.read()
old = 'from diffusers.optimization import (\n    Union, SchedulerType, Optional,\n    Optimizer, TYPE_TO_SCHEDULER_FUNCTION\n)'
if old in src:
    with open(p, 'w') as f:
        f.write(src.replace(old,
            'from typing import Union, Optional\nfrom torch.optim import Optimizer\n'
            'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'))
    print('Patched lr_scheduler.py')
else:
    print('lr_scheduler.py: OK')

# --- Patch 4: LANG_EMB_KEY fallback in 3 files ---
for name in ['dataset/lerobot_dataset.py', 'dataset/robomimic_hdf5_image_dataset.py', 'env/robomimic/robomimic_image_wrapper.py']:
    p = f'{DP}/{name}'
    if not os.path.exists(p): continue
    with open(p) as f: src = f.read()
    old_imp = 'from robomimic.macros import LANG_EMB_KEY'
    if old_imp in src and 'try:\n    from robomimic.macros' not in src:
        with open(p, 'w') as f:
            f.write(src.replace(old_imp,
                'try:\n    from robomimic.macros import LANG_EMB_KEY\n'
                'except ImportError:\n    LANG_EMB_KEY = "lang_emb"'))
        print(f'Patched {name}')
    else:
        print(f'{name}: OK')

print('All patches applied.')

## 6. Create OpenCabinet task config

In [ ]:
import subprocess

PROJECT = '/content/cs188-cabinet-door-project'
VENV_PY = f'{PROJECT}/.venv/bin/python'

result = subprocess.check_output([
    VENV_PY, '-c',
    'from robocasa.utils.dataset_registry_utils import get_ds_path; print(get_ds_path("OpenCabinet", source="human", split="pretrain"))'
], stderr=subprocess.DEVNULL).decode().strip()
dataset_path = result.split('\n')[-1]
print(f'Dataset path: {dataset_path}')

config = f'''name: OpenCabinet

shape_meta: &shape_meta
  obs:
    robot0_agentview_right_image:
      shape: [3, 256, 256]
      type: rgb
      lerobot_keys: ['video.robot0_agentview_right']
    robot0_agentview_left_image:
      shape: [3, 256, 256]
      type: rgb
      lerobot_keys: ['video.robot0_agentview_left']
    robot0_eye_in_hand_image:
      shape: [3, 256, 256]
      type: rgb
      lerobot_keys: ['video.robot0_eye_in_hand']
    robot0_base_to_eef_pos:
      shape: [3]
      lerobot_keys: ['state.end_effector_position_relative']
    robot0_base_to_eef_quat:
      shape: [4]
      lerobot_keys: ['state.end_effector_rotation_relative']
    robot0_gripper_qpos:
      shape: [2]
      lerobot_keys: ['state.gripper_qpos']
    lang_emb:
      shape: [768]
  action:
    shape: [12]
    lerobot_keys: ['action.end_effector_position', 'action.end_effector_rotation', 'action.gripper_close', 'action.base_motion', 'action.control_mode']

abs_action: &abs_action False

env_runner:
  _target_: diffusion_policy.env_runner.robomimic_image_runner.RobomimicImageRunner
  dataset_path: /mnt/nfs_client/robocasa/datasets/v1.0/test/atomic/PnPCounterToCabinet/20250811/demo.hdf5
  shape_meta: *shape_meta
  n_train: 0
  n_train_vis: 1
  train_start_idx: 0
  n_test: 50
  n_test_vis: 4
  test_start_seed: 100000
  max_steps: 600
  n_obs_steps: ${{n_obs_steps}}
  n_action_steps: ${{n_action_steps}}
  render_obs_key: 'robot0_agentview_right_image'
  fps: 10
  crf: 22
  past_action: ${{past_action_visible}}
  abs_action: *abs_action
  tqdm_interval_sec: 1.0
  n_envs: 1
  env_kwargs:
    seed: 1111111
    obj_instance_split: 'test'
    style_ids: null
    layout_ids: null
    layout_and_style_ids: [[1, 1], [2, 2], [3, 3], [4, 4], [5, 5], [6, 6], [7, 7], [8, 8], [9, 9], [10, 10]]
    clutter_mode: 1
    use_camera_obs: true
    use_object_obs: true
    camera_depths: false
    has_renderer: false
    has_offscreen_renderer: true
    camera_names: ['robot0_agentview_left', 'robot0_agentview_right', 'robot0_eye_in_hand']
    camera_widths: 256
    camera_heights: 256
    ignore_done: true
    reward_shaping: false

dataset:
  _target_: diffusion_policy.dataset.lerobot_dataset.LerobotCotrainingDataset
  shape_meta: *shape_meta
  dataset_paths:
    - {dataset_path}
  horizon: ${{horizon}}
  pad_before: ${{eval:'${{n_obs_steps}}-1+${{n_latency_steps}}'}}
  pad_after: ${{eval:'${{n_action_steps}}-1'}}
  n_obs_steps: ${{dataset_obs_steps}}
  abs_action: *abs_action
  rotation_rep: 'rotation_6d'
  use_legacy_normalizer: False
  use_cache: True
  seed: 42
  val_ratio: 0.02
'''

config_path = f'{PROJECT}/cabinet_door_project/diffusion_policy/diffusion_policy/config/task/robocasa/OpenCabinet.yaml'
with open(config_path, 'w') as f:
    f.write(config)
print(f'Config written to: {config_path}')

## 7. Verify setup

In [ ]:
!cd /content/cs188-cabinet-door-project/cabinet_door_project/diffusion_policy && \
    source /content/cs188-cabinet-door-project/.venv/bin/activate && \
    python -c "\
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); \
from diffusion_policy.dataset.lerobot_dataset import LerobotCotrainingDataset; \
from diffusion_policy.model.common.lr_scheduler import get_scheduler; \
from robomimic.macros import LANG_EMB_KEY; \
import robomimic.utils.lang_utils as LU; LU.LangEncoder(); \
print('All imports OK. Ready to train.')"

## 8. Train

Batch size 192 works on A100. For V100, add `dataloader.batch_size=64 val_dataloader.batch_size=64`.

In [ ]:
!cd /content/cs188-cabinet-door-project/cabinet_door_project/diffusion_policy && \
    source /content/cs188-cabinet-door-project/.venv/bin/activate && \
    wandb login && \
    python train.py \
        --config-name=train_diffusion_transformer_bs192 \
        task=robocasa/OpenCabinet \
        dataloader.num_workers=4 \
        val_dataloader.num_workers=4 \
        'hydra.run.dir=/content/drive/MyDrive/cs188_diffusion_policy/outputs/${now:%Y.%m.%d}/${now:%H.%M.%S}_OpenCabinet'

## 9. Resume training (if disconnected)

Re-run cells 1-7 to reinstall, then run this cell.

In [ ]:
!cd /content/cs188-cabinet-door-project/cabinet_door_project/diffusion_policy && \
    source /content/cs188-cabinet-door-project/.venv/bin/activate && \
    CKPT=$(ls -td /content/drive/MyDrive/cs188_diffusion_policy/outputs/*/*_OpenCabinet 2>/dev/null | head -1) && \
    echo "Resuming from: $CKPT" && \
    python train.py \
        --config-name=train_diffusion_transformer_bs192 \
        task=robocasa/OpenCabinet \
        dataloader.num_workers=4 \
        val_dataloader.num_workers=4 \
        training.resume=True \
        "hydra.run.dir=$CKPT"